# 01 – Data Cleaning

**Purpose:** Create a clean, merged dataset ready for feature engineering and modelling.

**Steps:**
1. Load raw datasets (crop yield, district soil lookup, GEE geo features).
2. Handle missing values.
3. Add state names to the soil lookup (using a district–state mapping).
4. Merge crop and soil data by state (or district if available).
5. Remove unrealistic or impossible values.
6. Save cleaned dataset as `cleaned_training_data.csv`.

In [47]:
import pandas as pd
import numpy as np
import re

# For displaying more rows/columns
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 100)

In [48]:
# Load crop yield data
crop = pd.read_csv('../data/crop_yield.csv')
print("Crop data shape:", crop.shape)

# Load district soil lookup (N, P, K, pH)
soil = pd.read_csv('../data/district_soil_lookup.csv')
print("Soil lookup shape:", soil.shape)

# Load GEE geo features (for reference, not used in cleaning)
geo = pd.read_csv('../data/india_geo_features.csv')
print("GEO data shape:", geo.shape)

Crop data shape: (19689, 10)
Soil lookup shape: (738, 6)
GEO data shape: (1132, 4)


In [49]:
soil.head(10)

,state_name,district_name,Nitrogen,Phosphorus,Potassium,Soil Ph
0,Andaman And Nicobar Islands,Nicobars,4.727811,4.727811,4.727811,4.727811
1,Andaman And Nicobar Islands,North And Middle Andaman,20.018349,20.018349,20.018349,20.018349
2,Andaman And Nicobar Islands,South Andamans,11.195876,11.195876,11.195876,11.195876
3,Andhra Pradesh,Alluri Sitharama Raju,2.685761,2.688889,2.690864,2.645267
4,Andhra Pradesh,Anakapalli,9.231490,9.246363,9.238927,9.229227
5,Andhra Pradesh,Ananthapuramu,11.105403,11.104262,11.103881,11.106164
6,Andhra Pradesh,Annamayya,11.323037,11.370227,11.432432,11.057057
7,Andhra Pradesh,Bapatla,17.438095,17.414603,17.438095,17.438095
8,Andhra Pradesh,Chittoor,8.226020,8.231055,8.231055,8.216216
9,Andhra Pradesh,Dr. B.R. Ambedkar Konaseema,16.268238,16.268238,16.272727,16.261504


In [50]:
print("Columns in soil:", soil.columns.tolist())
print("\nFirst few district names:")
print(soil['district_name'].head(10))

Columns in soil: ['state_name', 'district_name', 'Nitrogen', 'Phosphorus', 'Potassium', 'Soil Ph']

First few district names:
0                       Nicobars
1       North And Middle Andaman
2                 South Andamans
3          Alluri Sitharama Raju
4                     Anakapalli
5                  Ananthapuramu
6                      Annamayya
7                        Bapatla
8                       Chittoor
9    Dr. B.R. Ambedkar Konaseema
Name: district_name, dtype: object


### Adding state names to soil lookup

The `district_soil_lookup.csv` contains `district_name` but no `state_name`.  
We need to add a `state_name` column to merge with crop data (which has `State`).

**Approach:** Use a mapping CSV that links district names to states.
If not available, we can manually create a small mapping for the districts present.

For now, we will create a placeholder mapping using a rule‑based approach (e.g., extract state from district name if possible).  
A more accurate method would be to use a geospatial lookup – but for cleaning we will note the limitation.

In [51]:
# Try to extract state from district name (some districts contain state name in parentheses)
def extract_state_from_district(name):
    # Example: "Bhilwara (Rajasthan)" -> "Rajasthan"
    match = re.search(r'\(([^)]+)\)', name)
    if match:
        return match.group(1).strip()
    return None

soil['state_from_name'] = soil['district_name'].apply(extract_state_from_district)
print("Districts with state in parentheses:")
print(soil[soil['state_from_name'].notna()][['district_name', 'state_from_name']].head(10))

Districts with state in parentheses:
                              district_name state_from_name
101                         Kaimur (Bhabua)          Bhabua
146  Manendragarh-Chirmiri-Bharatpur(M C B)           M C B
343                    Khandwa (East Nimar)      East Nimar
344                   Khargone (West Nimar)      West Nimar


In [52]:
print(f"Districts with state extracted: {soil['state_from_name'].notna().sum()}")
print(f"Missing state info: {soil['state_from_name'].isna().sum()}")

# Show districts without state info
missing = soil[soil['state_from_name'].isna()]['district_name'].unique()
print("\nExample districts missing state:")
print(missing[:10])

Districts with state extracted: 4
Missing state info: 734

Example districts missing state:
['Nicobars' 'North And Middle Andaman' 'South Andamans'
 'Alluri Sitharama Raju' 'Anakapalli' 'Ananthapuramu' 'Annamayya'
 'Bapatla' 'Chittoor' 'Dr. B.R. Ambedkar Konaseema']


**Observation:** Most district names do not contain the state name in parentheses.  
We need a proper district‑to‑state mapping file.  

For the purpose of this cleaning notebook, we will:
- Keep the soil data as is (without state) for now.
- In the merge step, we will use **state‑level averages** of soil nutrients.
- To get state‑level averages, we need to know which state each district belongs to.

**Temporary solution:** Use a manual dictionary for the districts present.  
We will create a small mapping for the unique districts in our soil data.

In [53]:
# Get all unique district names in soil data
unique_districts = soil['district_name'].unique()
print(f"Number of unique districts in soil data: {len(unique_districts)}")

# We will manually assign state names using known district–state relationships.
# This is a placeholder – you should replace this with a complete mapping file.

# Example mapping for a few districts (you must extend this!)
district_state_map = {
    'North Delhi': 'Delhi',
    'Central Delhi': 'Delhi',
    'South Delhi': 'Delhi',
    # Add more as needed...
}

# For demonstration, we will create a dummy assignment: 
# Assign 'Unknown' to all, then override known ones.
soil['state_name'] = 'Unknown'
for district, state in district_state_map.items():
    soil.loc[soil['district_name'] == district, 'state_name'] = state

print("State assignment completed.")
print(soil['state_name'].value_counts())

Number of unique districts in soil data: 735
State assignment completed.
state_name
Unknown    738
Name: count, dtype: int64


**CRITICAL:** The manual mapping above is incomplete.  
For a real project, you must obtain a complete CSV that maps every Indian district to its state.  
You can download such a file from the internet (e.g., from data.gov.in or other open sources).  

For the rest of this notebook, we will assume that you have a proper mapping file named `district_state_map.csv` with columns `district_name` and `state_name`.  
We will load it and merge.

In [54]:
# Load the cities CSV to create district‑to‑state mapping
cities = pd.read_csv('../data/India Cities LatLng.csv')

# Standardise names: uppercase and strip spaces
cities['district'] = cities['city'].str.upper().str.strip()
cities['state'] = cities['admin_name'].str.upper().str.strip()

# Keep only unique district‑state pairs (take first occurrence if duplicate district names)
district_state_map = cities[['district', 'state']].drop_duplicates(subset='district', keep='first')
print(f"Mapping created with {len(district_state_map)} unique districts.")

# Load soil lookup
soil = pd.read_csv('../data/district_soil_lookup.csv')
soil['district_upper'] = soil['district_name'].str.upper().str.strip()

# Merge state names
soil = soil.merge(district_state_map, left_on='district_upper', right_on='district', how='left')
soil.rename(columns={'state': 'state_name'}, inplace=True)

# Drop the temporary columns
soil.drop(columns=['district_upper', 'district'], inplace=True, errors='ignore')

# Check results
print(f"Districts with state assigned: {soil['state_name'].notna().sum()} out of {len(soil)}")
print("Sample of enriched soil data:")
soil.head()

Mapping created with 188 unique districts.
Districts with state assigned: state_name    738
state_name     50
dtype: int64 out of 738
Sample of enriched soil data:


,state_name,district_name,Nitrogen,Phosphorus,Potassium,Soil Ph,state_name
0,Andaman And Nicobar Islands,Nicobars,4.727811,4.727811,4.727811,4.727811,NaN
1,Andaman And Nicobar Islands,North And Middle Andaman,20.018349,20.018349,20.018349,20.018349,NaN
2,Andaman And Nicobar Islands,South Andamans,11.195876,11.195876,11.195876,11.195876,NaN
3,Andhra Pradesh,Alluri Sitharama Raju,2.685761,2.688889,2.690864,2.645267,NaN
4,Andhra Pradesh,Anakapalli,9.231490,9.246363,9.238927,9.229227,NaN


In [55]:
# ============================================================
# 1. Create district‑to‑state mapping (no rows dropped)
# ============================================================
try:
    cities = pd.read_csv('../data/India Cities LatLng.csv')
    cities['district'] = cities['city'].str.upper().str.strip()
    cities['state'] = cities['admin_name'].str.upper().str.strip()
    mapping = cities[['district', 'state']].drop_duplicates(subset='district', keep='first')
    mapping = mapping.rename(columns={'district': 'district_name', 'state': 'state_name'})
    print(f"Mapping created. Shape: {mapping.shape}")
except Exception as e:
    print(f"Error loading mapping: {e}")
    mapping = pd.DataFrame(columns=['district_name', 'state_name'])

# ============================================================
# 2. Load soil data and add state names (keep all soil rows)
# ============================================================
soil = pd.read_csv('../data/district_soil_lookup.csv')
soil['district_upper'] = soil['district_name'].str.upper().str.strip()

# Remove any existing 'state_name' column to avoid duplicates
if 'state_name' in soil.columns:
    soil = soil.drop(columns=['state_name'])

# Merge mapping
soil = soil.merge(mapping, left_on='district_upper', right_on='district_name', how='left')
soil = soil.drop(columns=['district_name_y', 'district_upper'], errors='ignore')
if 'district_name_x' in soil.columns:
    soil = soil.rename(columns={'district_name_x': 'district_name'})

# Remove duplicate columns if any
soil = soil.loc[:, ~soil.columns.duplicated()]

# Ensure state_name is a Series (not DataFrame)
if 'state_name' in soil.columns and isinstance(soil['state_name'], pd.DataFrame):
    soil['state_name'] = soil['state_name'].iloc[:, 0]

# For districts without a state in the mapping, set state_name to 'UNKNOWN'
soil['state_name'] = soil['state_name'].fillna('UNKNOWN')
print(f"Soil after mapping: {soil.shape}")
print(soil[['district_name', 'state_name']].head(10))

# ============================================================
# 3. Average soil nutrients per state (including UNKNOWN)
# ============================================================
soil_state_avg = soil.groupby('state_name')[['Nitrogen', 'Phosphorus', 'Potassium', 'Soil Ph']].mean().reset_index()
soil_state_avg = soil_state_avg.rename(columns={'state_name': 'State'})
print("State‑level soil averages:")
print(soil_state_avg.head())

# ============================================================
# 4. Load crop data and merge (keep all crop rows)
# ============================================================
crop = pd.read_csv('../data/crop_yield.csv')
crop['State'] = crop['State'].str.upper().str.strip()

# Left merge: keep all crop rows, even if soil data missing
merged = crop.merge(soil_state_avg, on='State', how='left')
print(f"Merged shape: {merged.shape}")
print(f"Missing soil values after merge (before imputation):")
print(merged[['Nitrogen', 'Phosphorus', 'Potassium', 'Soil Ph']].isnull().sum())

# ============================================================
# 5. OPTIONAL: Fill missing soil values with global medians
#    (only if you want to avoid any NaNs for training)
# ============================================================
# For now, we do NOT drop rows. We keep NaNs as is.
# Later in feature engineering, you can decide to impute or drop.
# If you want to fill them now, uncomment the next lines:
#
# for col in ['Nitrogen', 'Phosphorus', 'Potassium', 'Soil Ph']:
#     median_val = merged[col].median()
#     merged[col] = merged[col].fillna(median_val)
#
# print("Missing values filled with global medians.")

# ============================================================
# 6. Save the cleaned dataset (with possible NaNs)
# ============================================================
merged.to_csv('../data/cleaned_training_data.csv', index=False)
print("Saved cleaned_training_data.csv (missing soil values still present).")
print("Shape:", merged.shape)

Mapping created. Shape: (188, 2)
Soil after mapping: (738, 6)
                 district_name state_name
0                     Nicobars    UNKNOWN
1     North And Middle Andaman    UNKNOWN
2               South Andamans    UNKNOWN
3        Alluri Sitharama Raju    UNKNOWN
4                   Anakapalli    UNKNOWN
5                Ananthapuramu    UNKNOWN
6                    Annamayya    UNKNOWN
7                      Bapatla    UNKNOWN
8                     Chittoor    UNKNOWN
9  Dr. B.R. Ambedkar Konaseema    UNKNOWN


State‑level soil averages:
            State   Nitrogen  Phosphorus  Potassium    Soil Ph
0  ANDHRA PRADESH   9.482801    9.483425   9.483082   9.463157
1           BIHĀR   6.985665    6.985254   6.983230   6.988114
2    CHHATTĪSGARH  55.523148   55.532407  55.546296  55.486111
3         GUJARĀT   7.583372    7.583372   7.583372   7.583372
4         HARYĀNA   0.047306   93.615246  93.622663  93.847811
Merged shape: (19689, 14)
Missing soil values after merge (before imputation):
Nitrogen      13769
Phosphorus    13769
Potassium     13769
Soil Ph       13769
dtype: int64
Saved cleaned_training_data.csv (missing soil values still present).
Shape: (19689, 14)


In [56]:
# Check the soil dataframe structure
print("Soil columns:", soil.columns.tolist())
print("\nData types:")
print(soil.dtypes)
print("\nShape of soil:", soil.shape)

# Check state_name column specifically
if 'state_name' in soil.columns:
    print("\nType of soil['state_name']:", type(soil['state_name']))
    print("Is it a DataFrame?", isinstance(soil['state_name'], pd.DataFrame))
    print("First few values:")
    print(soil['state_name'].head())
else:
    print("\nWARNING: 'state_name' column not found!")

Soil columns: ['district_name', 'Nitrogen', 'Phosphorus', 'Potassium', 'Soil Ph', 'state_name']

Data types:
district_name     object
Nitrogen         float64
Phosphorus       float64
Potassium        float64
Soil Ph          float64
state_name        object
dtype: object

Shape of soil: (738, 6)

Type of soil['state_name']: <class 'pandas.core.series.Series'>
Is it a DataFrame? False
First few values:
0    UNKNOWN
1    UNKNOWN
2    UNKNOWN
3    UNKNOWN
4    UNKNOWN
Name: state_name, dtype: object


In [57]:
# Remove rows where state_name is still missing
initial = len(soil)
soil = soil.dropna(subset=['state_name'])
print(f"Removed {initial - len(soil)} rows without state name.")

Removed 0 rows without state name.


In [58]:
# Standardise state names in crop data
crop['State_clean'] = crop['State'].str.upper().str.strip()

# Also trim spaces in other text columns
crop['Crop'] = crop['Crop'].str.strip()
crop['Season'] = crop['Season'].str.strip()

In [61]:
# First, average soil nutrients per state
soil_state_avg = soil.groupby('state_name').mean(numeric_only=True).reset_index()
soil_state_avg = soil_state_avg.rename(columns={'state_name': 'State_clean'})
print("State‑level soil averages:")
print(soil_state_avg.head(25))

# Merge
merged = crop.merge(soil_state_avg, on='State_clean', how='left')
print(f"Merged shape: {merged.shape}")
print(f"Missing soil values after merge: {merged[['Nitrogen', 'Phosphorus', 'Potassium', 'Soil Ph']].isnull().sum()}")

State‑level soil averages:
          State_clean    Nitrogen  Phosphorus   Potassium     Soil Ph
0      ANDHRA PRADESH    9.482801    9.483425    9.483082    9.463157
1               BIHĀR    6.985665    6.985254    6.983230    6.988114
2        CHHATTĪSGARH   55.523148   55.532407   55.546296   55.486111
3             GUJARĀT    7.583372    7.583372    7.583372    7.583372
4             HARYĀNA    0.047306   93.615246   93.622663   93.847811
5    HIMĀCHAL PRADESH    3.740137    3.747427    3.743997    3.659949
6   JAMMU AND KASHMĪR    3.698339    3.700935    3.700935    3.669263
7              KERALA    0.057692   47.096154   47.115385   47.083333
8      MADHYA PRADESH    9.249776    9.253832    9.258041    9.217031
9         MAHĀRĀSHTRA   13.010034   13.010357   13.012389   12.995245
10             ODISHA    9.257691    9.258412    9.258412    9.258051
11         PUDUCHERRY   10.957746   10.532203   10.478114   10.802083
12             PUNJAB    0.029514    6.129897    6.202814   38.

In [62]:
# States in crop data
crop_states = set(crop['State'].unique())
soil_states = set(soil_state_avg['State_clean'].unique())
missing_states = crop_states - soil_states
print(f"States in crop but missing in soil averages: {len(missing_states)}")
print(sorted(missing_states)[:20])  # show first 20

States in crop but missing in soil averages: 21
['ARUNACHAL PRADESH', 'ASSAM', 'BIHAR', 'CHHATTISGARH', 'DELHI', 'GOA', 'GUJARAT', 'HARYANA', 'HIMACHAL PRADESH', 'JAMMU AND KASHMIR', 'JHARKHAND', 'KARNATAKA', 'MAHARASHTRA', 'MANIPUR', 'MEGHALAYA', 'MIZORAM', 'NAGALAND', 'TAMIL NADU', 'TRIPURA', 'UTTARAKHAND']


In [63]:
# Re-load soil without any mapping overrides
soil_fixed = pd.read_csv('../data/district_soil_lookup.csv')
print("Original soil columns:", soil_fixed.columns.tolist())
print("Original state names sample:")
print(soil_fixed[['district_name', 'state_name']].head())

Original soil columns: ['state_name', 'district_name', 'Nitrogen', 'Phosphorus', 'Potassium', 'Soil Ph']
Original state names sample:
              district_name                   state_name
0                  Nicobars  Andaman And Nicobar Islands
1  North And Middle Andaman  Andaman And Nicobar Islands
2            South Andamans  Andaman And Nicobar Islands
3     Alluri Sitharama Raju               Andhra Pradesh
4                Anakapalli               Andhra Pradesh


In [64]:
soil_state_avg_fixed = soil_fixed.groupby('state_name')[['Nitrogen', 'Phosphorus', 'Potassium', 'Soil Ph']].mean().reset_index()
soil_state_avg_fixed = soil_state_avg_fixed.rename(columns={'state_name': 'State'})
# Standardise state names (uppercase, strip)
soil_state_avg_fixed['State'] = soil_state_avg_fixed['State'].str.upper().str.strip()
print("State‑level soil averages (fixed):")
print(soil_state_avg_fixed.head())
print(f"Number of states in soil averages: {len(soil_state_avg_fixed)}")

State‑level soil averages (fixed):
                         State   Nitrogen  Phosphorus  Potassium    Soil Ph
0  ANDAMAN AND NICOBAR ISLANDS  11.980679   11.980679  11.980679  11.980679
1               ANDHRA PRADESH  11.220345   11.221094  11.216321  11.186357
2            ARUNACHAL PRADESH  10.110119    9.830129   9.936550  10.067846
3                        ASSAM  50.542993   50.533311  50.482154  50.574680
4                        BIHAR   7.021569    7.021058   7.019740   7.038964
Number of states in soil averages: 32


In [65]:
crop = pd.read_csv('../data/crop_yield.csv')
crop['State'] = crop['State'].str.upper().str.strip()

merged_fixed = crop.merge(soil_state_avg_fixed, on='State', how='left')
print(f"Merged shape: {merged_fixed.shape}")
print(f"Missing soil values after merge:")
print(merged_fixed[['Nitrogen', 'Phosphorus', 'Potassium', 'Soil Ph']].isnull().sum())

Merged shape: (19689, 14)
Missing soil values after merge:
Nitrogen      203
Phosphorus    203
Potassium     203
Soil Ph       203
dtype: int64


In [66]:
for col in ['Nitrogen', 'Phosphorus', 'Potassium', 'Soil Ph']:
    median_val = merged_fixed[col].median()
    merged_fixed[col] = merged_fixed[col].fillna(median_val)
    print(f"{col}: filled missing with {median_val:.2f}")

Nitrogen: filled missing with 11.22
Phosphorus: filled missing with 13.96
Potassium: filled missing with 14.75
Soil Ph: filled missing with 14.63


In [67]:
merged_fixed.to_csv('../data/cleaned_training_data.csv', index=False)
print("Saved cleaned_training_data.csv with original soil state names.")
print("Final shape:", merged_fixed.shape)
print("Missing values check:", merged_fixed[['Nitrogen', 'Phosphorus', 'Potassium', 'Soil Ph']].isnull().sum().sum())

Saved cleaned_training_data.csv with original soil state names.
Final shape: (19689, 14)
Missing values check: 0
